In [ ]:
import mpbn
import pandas as pd
import numpy as np
from pathlib import Path
import shutil

models_dir = Path("/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble")
data_file = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/data_binarized_tf_gene_sep.csv"
data = pd.read_csv(data_file,index_col=0)

KO 

In [28]:
## KO CEBPA

# Parameters
ko_gene = "CEBPA"

# Macrostates
data_file = "/home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/data_binarized_tf_gene_sep.csv"
data = pd.read_csv(data_file,index_col=0)

selected_models_ko = []

for model_file in models_dir.glob("*.bnet"):

    try:
        bn = mpbn.MPBooleanNetwork(str(model_file))
        atts = list(bn.attractors())
        #print(atts[:3])

        # KO
        bn[ko_gene] = "0"

        # Attractors
        attractors = pd.DataFrame(
            list(bn.attractors()))
        #print(model_file.name, len(attractors))

        if len(attractors) == 0:
            print("no attractors")

        # Common genes
        common_genes = attractors.columns.intersection(data.columns)
        # Subset
        data_subset = data[common_genes]
        attractors = attractors[common_genes]
        attractors = attractors[data_subset.columns]

        # names
        attractors.index = [f"A{i}" for i in range(len(attractors))]

        # Similarity

        similarity = pd.DataFrame(index=attractors.index)

        for state in data_subset.index:
            row = data_subset.loc[state]
            mask = row.notna()

            similarity[state] = (
                attractors.loc[:, mask]
                .eq(row[mask], axis=1)
                .mean(axis=1))

        # Condition 1 : for all the attractors, S3 > S2

        cond1 = (similarity["S3"] > similarity["S2"]).all()

        if cond1:
            selected_models_ko.append(model_file.name)
        

    except Exception as e:
        print(model_file.name, e)

print(f"Nb of models selected KO : {len(selected_models_ko)}")

# KI CEBPA
ki_gene = "CEBPA"

selected_models_ki = []
data = pd.read_csv(data_file,index_col=0)

for model_file in models_dir.glob("*.bnet"):

    try:
        bn = mpbn.MPBooleanNetwork(str(model_file))
        atts = list(bn.attractors())
        #print(atts[:3])

        # KI
        bn[ki_gene] = "1"

        # Attractors
        attractors = pd.DataFrame(
            list(bn.attractors()))
        #print(model_file.name, len(attractors))

        if len(attractors) == 0:
            print("no attractors")

        # Common genes
        common_genes = attractors.columns.intersection(data.columns)
        # Subset
        data_subset = data[common_genes]
        attractors = attractors[common_genes]
        attractors = attractors[data_subset.columns]

        # names
        attractors.index = [f"A{i}" for i in range(len(attractors))]

        # Similarity
        similarity = pd.DataFrame(index=attractors.index)

        for state in data_subset.index:
            row = data_subset.loc[state]
            mask = row.notna()

            similarity[state] = (
                attractors.loc[:, mask]
                .eq(row[mask], axis=1)
                .mean(axis=1))

        # Condition 1 : for all the attractors, S2 > S3
        cond1 = (similarity["S2"] > similarity["S3"]).all()

        if cond1:
            selected_models_ki.append(model_file.name)
        

    except Exception as e:
        print(model_file.name, e)

print(f"Nb of models selected KI : {len(selected_models_ki)}")

# Intersection between KO and KI CEBPA
models_cebpa = list(set(selected_models_ko) & set(selected_models_ki))
print(f"Nb of models selected : {len(models)}")

Nb of models selected KO : 178
Nb of models selected KI : 182
Nb of models selected : 0


In [34]:
def screen_models(models_dir, data, gene, value, direction):
    """
    Screen Boolean models after a KO/KI perturbation.

    Parameters
    ----------
    models_dir : Path
        Folder containing .bnet models.
    data : pd.DataFrame
        Binarized macrostates.
    gene : str
        Gene to perturb.
    value : str
        "0" for KO, "1" for KI.

    Returns
    -------
    selected_models : list
        List of selected model names.
    """

    selected_models = []

    for model_file in models_dir.glob("*.bnet"):

        try:
            # Load model
            bn = mpbn.MPBooleanNetwork(str(model_file))

            # Perturbation
            bn[gene] = value

            # Compute attractors
            attractors = pd.DataFrame(list(bn.attractors()))

            if attractors.empty:
                continue

            # Keep common genes
            common_genes = attractors.columns.intersection(data.columns)

            if len(common_genes) == 0:
                continue

            attractors = attractors[common_genes]
            data_subset = data[common_genes]

            attractors = attractors[data_subset.columns]
            attractors.index = [f"A{i}" for i in range(len(attractors))]

            # Similarity matrix
            similarity = pd.DataFrame(index=attractors.index)

            for state in data_subset.index:
                row = data_subset.loc[state]
                mask = row.notna()

                similarity[state] = (
                    attractors.loc[:, mask]
                    .eq(row[mask], axis=1)
                    .mean(axis=1)
                )

            # Selection criterion
            if direction == "KO":
                cond1 = (similarity["S3"] > similarity["S2"]).all()

            elif direction == "KI":
                cond1 = (similarity["S2"] > similarity["S3"]).all()

            else:
                raise ValueError("direction must be 'KO' or 'KI'")

            if cond1:
                selected_models.append(model_file.name)

        except Exception as e:
            print(model_file.name, e)

    return selected_models

In [40]:
data = pd.read_csv(data_file, index_col=0)

selected_models_ko = screen_models(models_dir=models_dir,data=data,gene="CEBPA",value="0",direction="KO")

selected_models_ki = screen_models(models_dir=models_dir,data=data,gene="CEBPA",value="1",direction="KI")

#Intersection KO and Ki 
models_cebpa = list(set(selected_models_ko) & set(selected_models_ki))

print(f"KO : {len(selected_models_ko)}")
print(f"KI : {len(selected_models_ki)}")
print(f"Intersection : {len(models_cebpa)}")

KO : 178
KI : 182
Intersection : 178


Sort the templates and place the selected ones in a folder

In [41]:
# New folder
selected_dir = models_dir / "models_selected"
selected_dir.mkdir(exist_ok=True)

# Copy models selected
for model_name in models_cebpa:
    src = models_dir / model_name
    dst = selected_dir / model_name

    if src.exists():
        shutil.copy2(src, dst)
    else:
        print(f"File no found : {model_name}")

print(f"{len(models_cebpa)} modeles paste : {selected_dir}")

178 modeles paste : /home/a.blanc-boekholt/Documents/Singlecell-R/Scripts/Boolean_model_BoNesis/Bonesis_test/ensemble/models_selected


Intersection KO and KI CEBPA and SPI1

In [37]:
data = pd.read_csv(data_file, index_col=0)

selected_models_ko = screen_models(models_dir=models_dir,data=data,gene="SPI1",value="0",direction="KO")

selected_models_ki = screen_models(models_dir=models_dir,data=data,gene="SPI1",value="1",direction="KI")

#Intersection KO and Ki 
models_spi1 = list(set(selected_models_ko) & set(selected_models_ki))

print(f"KO : {len(selected_models_ko)}")
print(f"KI : {len(selected_models_ki)}")
print(f"Intersection : {len(models_spi1)}")

KO : 80
KI : 82
Intersection : 80


In [38]:
# Intersection betwenn
models = list(set(models_cebpa) & set(models_spi1))
print(f"Nb of models selected : {len(models)}")

Nb of models selected : 80
